# 01 — Explore Streams

Visualize the 7 target Milky Way stellar streams from Gaia DR3.
For each stream: density profile along phi1, proper motion coherence, and CMD.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'config' / 'streams.yaml') as f:
    streams_cfg = yaml.safe_load(f)

STREAMS = list(streams_cfg['streams'].keys())
H5_PATH = ROOT / 'data' / 'processed' / 'streams.h5'
print(f'Streams: {STREAMS}')
print(f'HDF5 path: {H5_PATH}')

In [ ]:
# Load all streams
stream_data = {}
with h5py.File(str(H5_PATH), 'r') as f:
    for name in STREAMS:
        grp = f[f'streams/{name}/members']
        stream_data[name] = {
            'phi1': grp['phi1'][:],
            'phi2': grp['phi2'][:],
            'dist': grp['dist'][:],
            'pm1': grp['pm1'][:],
            'pm2': grp['pm2'][:],
        }
        print(f'{name:8s}: {len(grp["phi1"])} members')

## Sky positions (phi1, phi2) in stream frame

In [ ]:
fig, axes = plt.subplots(len(STREAMS), 1, figsize=(12, 3*len(STREAMS)))
for ax, name in zip(axes, STREAMS):
    d = stream_data[name]
    ax.scatter(d['phi1'], d['phi2'], s=1, alpha=0.4)
    ax.set_xlabel('phi1 [deg]')
    ax.set_ylabel('phi2 [deg]')
    ax.set_title(f'{name} ({len(d["phi1"])} stars)')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Density profiles along phi1

In [ ]:
fig, axes = plt.subplots(len(STREAMS), 1, figsize=(12, 2.5*len(STREAMS)))
for ax, name in zip(axes, STREAMS):
    d = stream_data[name]
    ax.hist(d['phi1'], bins=100, color='steelblue', alpha=0.7)
    ax.set_xlabel('phi1 [deg]')
    ax.set_ylabel('N stars')
    ax.set_title(f'{name} density profile')
plt.tight_layout()
plt.show()

## Proper motion vectors

In [ ]:
fig, axes = plt.subplots(len(STREAMS), 1, figsize=(12, 3*len(STREAMS)))
for ax, name in zip(axes, STREAMS):
    d = stream_data[name]
    # Subsample for clarity
    idx = np.random.choice(len(d['phi1']), min(500, len(d['phi1'])), replace=False)
    ax.quiver(d['phi1'][idx], d['phi2'][idx],
              d['pm1'][idx], d['pm2'][idx],
              scale=50, alpha=0.5, width=0.002)
    ax.set_xlabel('phi1 [deg]')
    ax.set_ylabel('phi2 [deg]')
    ax.set_title(f'{name} proper motion vectors')
plt.tight_layout()
plt.show()

## Stream summary statistics

In [ ]:
import pandas as pd
rows = []
for name in STREAMS:
    d = stream_data[name]
    rows.append({
        'stream': name,
        'n_members': len(d['phi1']),
        'phi1_min': d['phi1'].min(),
        'phi1_max': d['phi1'].max(),
        'phi2_std': d['phi2'].std(),
        'dist_median_kpc': np.median(d['dist']),
        'pm1_mean': d['pm1'].mean(),
        'pm1_std': d['pm1'].std(),
    })
pd.DataFrame(rows).round(3)